# Exercise 1: Machine Learning Basics

**Unit:** Unit 3: ML Basics — regression vs classification, the XOR problem,
gradient descent, and model interpretability.

## 📝 Instructions

Each exercise gives you a **working baseline** you can run immediately, followed by a
**🖊️ Your turn** extension marked with `# TODO`. The notebook runs clean top-to-bottom
before you change anything — your job is to extend it.

**Steps:**
1. Run each baseline cell and read its output.
2. Complete the 🖊️ Your turn tasks in the same cell.
3. Re-run and check your results.
4. Solutions are released by your instructor.

---

## Exercise 1: Regression vs Classification

Same framing as `01_regression_classification.ipynb`: regression predicts a **number**,
classification predicts a **category** — and each has its own metrics. We run both on the
*same* real patients so the only thing that changes is the question we ask.

In [1]:
# Exercise 1: regression vs classification on the SAME real patients
# WHY the same data twice: it shows that "regression or classification?" is a question
# about the TARGET you choose, not about the dataset you were handed.
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score

print("Exercise 1: Regression vs Classification")
print("-" * 60)

# Real data: 442 diabetes patients (Efron et al., 2004), bundled with scikit-learn.
diabetes = load_diabetes(scaled=False)
bmi = diabetes.data[:, list(diabetes.feature_names).index("bmi")].reshape(-1, 1)
progression = diabetes.target.astype(float)
print(f"  {len(bmi)} real patients: body mass index -> disease progression after one year")

# --- Regression: predict the progression SCORE from body mass index ---
X_train, X_test, y_train, y_test = train_test_split(bmi, progression, test_size=0.2, random_state=0)
reg = LinearRegression().fit(X_train, y_train)
mse = mean_squared_error(y_test, reg.predict(X_test))
print(f"  Regression  — test MSE: {mse:.1f}, test R^2: {reg.score(X_test, y_test):.3f}")

# --- Classification: same feature, but now ask a YES/NO question ---
# "Did this patient's disease progress more than the median patient's?"
high = (progression >= np.median(progression)).astype(int)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(bmi, high, test_size=0.2, random_state=0)
clf = LogisticRegression().fit(Xc_train, yc_train)
acc = accuracy_score(yc_test, clf.predict(Xc_test))
print(f"  Classification — test accuracy: {acc:.3f}  "
      f"(coin-flip baseline on a median split: 0.500)")

# 🖊️ Your turn:
# TODO 1: Print one regression prediction (e.g. reg.predict([[30.0]])) and one
#         classification prediction for the same BMI of 30. How do the two
#         answers differ in KIND, not just value?
# TODO 2: Both models see ONE feature. Rebuild them with two columns — bmi and 's5'
#         (a blood serum measurement) — and re-run. Which metric improves more,
#         R^2 or accuracy, and why can they disagree?

Exercise 1: Regression vs Classification
------------------------------------------------------------
  442 real patients: body mass index -> disease progression after one year
  Regression  — test MSE: 4150.7, test R^2: 0.191
  Classification — test accuracy: 0.629  (coin-flip baseline on a median split: 0.500)


## Exercise 2: The XOR Problem and Linear Separability

A **linear** model cannot separate XOR (`02_perceptron_xor.ipynb`). But watch what a
single hand-crafted feature does.

In [2]:
# Exercise 2: XOR — a linear model fails, a crafted feature fixes it
import numpy as np
from sklearn.linear_model import LogisticRegression

print("Exercise 2: XOR and Linear Separability")
print("-" * 60)

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

# Baseline: a purely linear model on the raw inputs
# (C=100 weakens sklearn's default regularization, which otherwise
#  keeps the weights too small for a clean fit on just 4 points)
linear_clf = LogisticRegression(C=100).fit(X_xor, y_xor)
linear_acc = linear_clf.score(X_xor, y_xor)
print(f"  Linear model on raw (x1, x2):        accuracy = {linear_acc:.2f}")
print("  A single line cannot put (0,1) and (1,0) on one side and")
print("  (0,0) and (1,1) on the other - XOR is not linearly separable.")

# Baseline: add the product feature x1*x2 and try again
X_feat = np.column_stack([X_xor, X_xor[:, 0] * X_xor[:, 1]])
feat_clf = LogisticRegression(C=100).fit(X_feat, y_xor)
feat_acc = feat_clf.score(X_feat, y_xor)
print(f"\n  Same model + crafted feature x1*x2:  accuracy = {feat_acc:.2f}")
print("  The extra feature bends the space so one line suffices -")
print("  hidden layers in a neural network learn such features automatically")
print("  (that is what 03_solving_xor_keras.ipynb does).")

# 🖊️ Your turn:
# TODO 1: Try a different crafted feature: (x1 - x2)**2. Does it also reach 1.00?
# TODO 2: Print the model predictions for all four XOR rows in both versions
#         (linear_clf needs the 2-column input, feat_clf the 3-column one).


Exercise 2: XOR and Linear Separability
------------------------------------------------------------
  Linear model on raw (x1, x2):        accuracy = 0.50
  A single line cannot put (0,1) and (1,0) on one side and
  (0,0) and (1,1) on the other - XOR is not linearly separable.

  Same model + crafted feature x1*x2:  accuracy = 1.00
  The extra feature bends the space so one line suffices -
  hidden layers in a neural network learn such features automatically
  (that is what 03_solving_xor_keras.ipynb does).


## Exercise 3: Gradient Descent by Hand

Like `04_gradient_descent_loss_functions.ipynb`: fit `y = w * x` by repeatedly stepping
`w` against the gradient of the MSE loss — here on 50 real US states, with no "true w"
to check against. The only judge is whether the loss keeps going down.

In [3]:
# Exercise 3: gradient descent for a one-parameter model y = w * x, on REAL data
# WHY no intercept: with a single parameter you can watch the whole optimization in one
# column of numbers — and on real data there is no planted answer to peek at.
import numpy as np
import pandas as pd

print("Exercise 3: Gradient Descent")
print("-" * 60)

# Real data: US state crime statistics (USArrests, 50 states, 1973 rates per 100,000).
crime = pd.read_csv("../../../Course 04/datasets/raw/crime_statistics_original_50.csv")
x = crime["Assault"].to_numpy(dtype=float) / 100.0   # assaults per 100k, in HUNDREDS
y = crime["Murder"].to_numpy(dtype=float)            # murders per 100k
print(f"  {len(x)} US states | assaults/100k: {x.min()*100:.0f}-{x.max()*100:.0f}, "
      f"murders/100k: {y.min():.1f}-{y.max():.1f}")

def mse_loss(w):
    return np.mean((y - w * x) ** 2)

def gradient(w):
    # d/dw of mean((y - w*x)^2) = -2 * mean(x * (y - w*x))
    return -2 * np.mean(x * (y - w * x))

w = 0.0            # start from "assaults tell us nothing about murders"
learning_rate = 0.02

print(f"  {'step':>4s} {'w':>8s} {'loss':>10s}")
for step in range(31):
    if step % 5 == 0:
        print(f"  {step:4d} {w:8.4f} {mse_loss(w):10.4f}")
    w = w - learning_rate * gradient(w)

# The closed-form answer for a one-parameter model, to check the loop against.
w_exact = np.sum(x * y) / np.sum(x * x)
print(f"\n  Final w = {w:.4f}; exact least-squares w = {w_exact:.4f}")
print(f"  Reading it: about {w:.1f} murders per 100k for every 100 assaults per 100k.")
print(f"  The loss stops at {mse_loss(w):.2f}, not 0 — assault rates explain a lot")
print("  about murder rates, but nowhere near everything.")

# 🖊️ Your turn:
# TODO 1: Set learning_rate = 0.3 and re-run. What happens to the loss column, and why?
#         (Hint: compute 2 * np.mean(x ** 2) — steps larger than 2 divided by that number
#         overshoot the minimum by more than the distance to it.)
# TODO 2: Set learning_rate = 0.001. How many steps would you now need, roughly?
# TODO 3: Add an intercept (y = w*x + b) and update both w and b each step. Does the
#         final loss drop? What does a non-zero b mean for a state with no assaults?

Exercise 3: Gradient Descent
------------------------------------------------------------
  50 US states | assaults/100k: 45-337, murders/100k: 0.8-17.4
  step        w       loss
     0   0.0000    79.2440
     5   2.4252    22.0576
    10   3.5407     9.9590
    15   4.0538     7.3993
    20   4.2898     6.8578
    25   4.3983     6.7432
    30   4.4483     6.7190

  Final w = 4.4544; exact least-squares w = 4.4908
  Reading it: about 4.5 murders per 100k for every 100 assaults per 100k.
  The loss stops at 6.72, not 0 — assault rates explain a lot
  about murder rates, but nowhere near everything.


## Exercise 4: Interpretability — Which Features Matter?

`05_model_interpretability_shap_lime.ipynb` used SHAP and LIME; here you check the
simplest interpretability signal — impurity-based feature importances. The trick: take
**real** measurements and bolt on two columns of pure random numbers. Nobody has to
guess which columns are meaningless — we added them ourselves, so the ranking has a
right answer even though the data is real.

In [4]:
# Exercise 4: feature importances on REAL measurements plus two planted decoys
# WHY inject noise into real data: the tumour features carry genuine signal we did not
# invent, while the two random columns give us a known-worthless baseline to compare against.
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

print("Exercise 4: Interpretability")
print("-" * 60)

cancer = load_breast_cancer()
real_pair = ["mean radius", "mean concave points"]
cols = [list(cancer.feature_names).index(f) for f in real_pair]
X_real = cancer.data[:, cols]
y = cancer.target                      # 0 = malignant, 1 = benign
n = len(y)

# Two decoy columns of pure noise, drawn to look like plausible measurements.
rng = np.random.default_rng(7)
decoys = rng.normal(size=(n, 2))

X = np.column_stack([X_real, decoys])
feature_names = real_pair + ["decoy_1", "decoy_2"]
print(f"  {n} real biopsies | 2 real measurements + 2 planted noise columns")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=7)
forest = RandomForestClassifier(n_estimators=100, random_state=7).fit(X_train, y_train)
print(f"  Test accuracy: {forest.score(X_test, y_test):.3f}\n")

print("  Impurity-based feature importances (the real measurements should rank on top):")
order = np.argsort(forest.feature_importances_)[::-1]
for idx in order:
    print(f"    {feature_names[idx]:20s} {forest.feature_importances_[idx]:.3f}")
print("\n  Note the decoys do NOT score 0. Impurity importance rewards any column a tree")
print("  can split on, so pure noise still collects a little credit — that is a limitation")
print("  of the method, not evidence that the noise means something.")

# 🖊️ Your turn:
# TODO 1: Add a third decoy column and re-run. Do the decoys keep roughly the same
#         combined share of the importance, or does each new one steal more?
# TODO 2: Add a fifth feature that is a COPY of "mean radius". What happens to the
#         importance previously assigned to "mean radius", and why is that a warning
#         about reading importances too literally?

Exercise 4: Interpretability
------------------------------------------------------------
  569 real biopsies | 2 real measurements + 2 planted noise columns
  Test accuracy: 0.909

  Impurity-based feature importances (the real measurements should rank on top):
    mean concave points  0.536
    mean radius          0.344
    decoy_2              0.062
    decoy_1              0.059

  Note the decoys do NOT score 0. Impurity importance rewards any column a tree
  can split on, so pure noise still collects a little credit — that is a limitation
  of the method, not evidence that the noise means something.


---

## ✅ Check Your Work

- ✅ Does the whole notebook still run top-to-bottom after your edits?
- ✅ Exercise 2: can you say in one sentence why the crafted feature fixes XOR?
- ✅ Exercise 3: can you explain what a too-large learning rate did to the loss?
- ✅ Exercise 4: did the two real measurements outrank both planted decoys — and can you say why the decoys still scored above zero?

**Next steps:**
- Solutions are released by your instructor.
- Take the quiz in `../quizzes/` and continue to Unit 4: `../../unit4-neural-networks-basics/README.md`.